In [1]:
pip install -q gradio ultralytics torch torchvision opencv-python-headless pillow numpy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import gradio as gr
from PIL import Image
from torchvision import transforms
from torchvision.models import efficientnet_b0
from ultralytics import YOLO

# Model files
YOLO_PATH = "best.pt"
CLASSIFIER_PATH = "efficientnet_freshness.pt"  

YOLO_CONF = 0.4
ETHYLENE_THRESHOLD = 3.5   # ppm
METHANE_THRESHOLD = 18     # ppm
MIN_CROP = 40              # ignore tiny boxes (pixels)

CLASS_NAMES = [
    "FreshApple","FreshBanana","FreshBellpepper","FreshCarrot","FreshCucumber",
    "FreshMango","FreshOrange","FreshPotato","FreshStrawberry","FreshTomato",
    "RottenApple","RottenBanana","RottenBellpepper","RottenCarrot","RottenCucumber",
    "RottenMango","RottenOrange","RottenPotato","RottenStrawberry","RottenTomato"
]

for p in (YOLO_PATH, CLASSIFIER_PATH):
    print(p, "->", "found" if os.path.exists(p) else "MISSING, upload it next to the notebook")

/workspaces/AccuKnox/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/codespace/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
best.pt -> found
efficientnet_freshness.pt -> found


In [ ]:
yolo_model = YOLO(YOLO_PATH)
classifier = efficientnet_b0(weights=None)
classifier.classifier[1] = nn.Linear(1280, 20)
classifier.load_state_dict(torch.load(CLASSIFIER_PATH, map_location="cpu"))
classifier.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

print("Models loaded successfully")

Models loaded successfully


In [ ]:
def classify_crop(crop_rgb):
    """Run EfficientNet-B0 on one RGB crop. Returns (fruit, freshness, confidence%)."""
    img = transform(Image.fromarray(crop_rgb)).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(classifier(img), dim=1)
        pred = probs.argmax(1).item()
        conf = probs[0][pred].item() * 100

    label = CLASS_NAMES[pred]
    if label.startswith("Fresh"):
        return label.replace("Fresh", ""), "Fresh", conf
    return label.replace("Rotten", ""), "Rotten", conf


def analyze(image, use_gas, ethylene, methane):
    if image is None:
        return None, "Please upload an image."

    display = image.copy()                        
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)  
    gas_rotten = use_gas and (ethylene > ETHYLENE_THRESHOLD or methane > METHANE_THRESHOLD)

    results = yolo_model(bgr, conf=YOLO_CONF, imgsz=640, verbose=False)
    boxes = results[0].boxes if results and results[0].boxes is not None else []

    report = []
    h, w = image.shape[:2]

    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        crop = image[y1:y2, x1:x2]

        if crop.shape[0] < MIN_CROP or crop.shape[1] < MIN_CROP:
            continue

        fruit, freshness, conf = classify_crop(crop)
        final = "Rotten" if (freshness == "Rotten" or gas_rotten) else "Fresh"

        color = (0, 200, 0) if final == "Fresh" else (255, 0, 0)  # RGB
        cv2.rectangle(display, (x1, y1), (x2, y2), color, 3)
        cv2.putText(display, f"{fruit} | {final} | {conf:.1f}%",
                    (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, color, 2)

        report.append(f"{fruit}: {final} (CNN: {freshness}, {conf:.1f}% confidence)")

    if not report:
        fruit, freshness, conf = classify_crop(image)
        final = "Rotten" if (freshness == "Rotten" or gas_rotten) else "Fresh"
        report.append(f"No box detected, classified whole image -> "
                      f"{fruit}: {final} ({conf:.1f}%)")

    if use_gas:
        report.append(f"\nEthylene: {ethylene} ppm | Methane: {methane} ppm "
                      f"-> Gas flag: {'Rotten' if gas_rotten else 'Fresh'}")

    return display, "\n".join(report)

In [7]:
demo = gr.Interface(
    fn=analyze,
    inputs=[
        gr.Image(type="numpy", label="Upload fruit or vegetable image"),
        gr.Checkbox(label="Simulate gas sensor fusion", value=False),
        gr.Slider(0, 10, value=2.0, step=0.1, label="Ethylene (ppm)"),
        gr.Slider(0, 40, value=10, step=0.5, label="Methane (ppm)"),
    ],
    outputs=[
        gr.Image(label="Detection result"),
        gr.Textbox(label="Freshness report", lines=6),
    ],
    title="Fruit & Vegetable Freshness Detection",
    description="YOLOv8n detects the produce, then EfficientNet-B0 classifies it as Fresh or Rotten. "
                "Optionally tick the gas box to simulate the MQ2/MQ4 sensor fusion.",
)


demo.launch(share=True) 
demo.launch(inline=True) 
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://ae512318abc149354d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.


In [6]:
   demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* Running on public URL: https://cb52a8efd52eeee078.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
